<a href="https://www.kaggle.com/code/avikdas567/arc-agi-3-hybrid-dsl-program-search-with-planning?scriptVersionId=306664604" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import sys
import os
import types
import random
import copy
import pandas as pd

# PATH SETUP

BASE_PATH = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents"
sys.path.append(BASE_PATH)

print("Repo path added.")

# PATCH AGENTS

fake_agents = types.ModuleType("agents")

class DummySwarm:
    def __init__(self, agent):
        self.agent = agent

    def main(self):
        print("Running simplified ARC loop...")
        frames = []
        dummy_frame = {
            "status": "GAME_OVER",
            "grid": [[0]],
            "available_actions": ["RESET"]
        }
        self.agent.choose_action(frames, dummy_frame)
        print("Finished dummy run.")

fake_agents.Swarm = DummySwarm
fake_agents.AVAILABLE_AGENTS = {}
sys.modules["agents"] = fake_agents

# FIX tracing
fake_tracing = types.ModuleType("agents.tracing")

def dummy_initialize(*args, **kwargs):
    return None

fake_tracing.initialize = dummy_initialize
sys.modules["agents.tracing"] = fake_tracing

from main import run_agent

# DSL OPERATIONS

def identity(x): return x

def rot90(x): return [list(row) for row in zip(*x[::-1])]
def rot180(x): return rot90(rot90(x))
def rot270(x): return rot90(rot180(x))

def flip_h(x): return [row[::-1] for row in x]
def flip_v(x): return x[::-1]

def crop(x):
    coords = [(i,j) for i in range(len(x)) for j in range(len(x[0])) if x[i][j] != 0]
    if not coords:
        return x
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    return [row[min(ys):max(ys)+1] for row in x[min(xs):max(xs)+1]]

DSL_OPS = [identity, rot90, rot180, rot270, flip_h, flip_v, crop]

# PROGRAM EXECUTION

def apply_program(grid, program):
    g = copy.deepcopy(grid)
    try:
        for op in program:
            g = op(g)
        return g
    except:
        return None

def score_program(program, examples):
    score = 0

    for inp, out in examples:
        pred = apply_program(inp, program)
        if pred is None:
            continue

        if pred == out:
            score += 5
        else:
            match = 0
            total = 0
            for i in range(len(out)):
                for j in range(len(out[0])):
                    total += 1
                    if i < len(pred) and j < len(pred[0]):
                        if pred[i][j] == out[i][j]:
                            match += 1
            score += match / max(total,1)

    return score

# PROGRAM SEARCH

def search_program(examples, max_depth=3, beam_width=5):

    beam = [[]]

    for _ in range(max_depth):
        candidates = []

        for prog in beam:
            for op in DSL_OPS:
                new_prog = prog + [op]
                s = score_program(new_prog, examples)
                candidates.append((s, new_prog))

        candidates.sort(key=lambda x: -x[0])
        beam = [p for _, p in candidates[:beam_width]]

    return beam

# FINAL AGENT

class FinalAgent:

    def __init__(self):
        self.Q = {}
        self.T = {}
        self.visits = {}
        self.last_state = None
        self.last_action = None

    def is_done(self, frames, latest_frame):
        return latest_frame["status"] != "NOT_FINISHED" or len(frames) > 300

    def hash(self, g): return str(g)

    def change(self, g1, g2):
        if g1 is None or g2 is None:
            return 0
        diff = 0
        total = 0
        for i in range(len(g1)):
            for j in range(len(g1[0])):
                total += 1
                if g1[i][j] != g2[i][j]:
                    diff += 1
        return diff / max(total,1)

    def rollout(self, state, actions, depth=2):
        total = 0
        current = state

        for _ in range(depth):
            best = max(actions, key=lambda a: self.Q.get(a,0))
            total += self.Q.get(best,0)
            next_s = self.T.get((current,best))
            if not next_s:
                break
            current = next_s

        return total

    def choose_action(self, frames, latest_frame):

        grid = latest_frame["grid"]
        state = self.hash(grid)
        actions = latest_frame["available_actions"]

        if not actions:
            return "RESET"

        if len(frames) >= 3:
            examples = []
            for i in range(len(frames)-1):
                examples.append((frames[i]["grid"], frames[i+1]["grid"]))

            programs = search_program(examples)
            best_prog = programs[0] if programs else None

            if best_prog:
                pred = apply_program(grid, best_prog)
                if pred and pred != grid:
                    return random.choice(actions)

        if self.last_state and self.last_action:
            prev = frames[-2]["grid"] if len(frames)>=2 else None
            c = self.change(prev, grid)

            r = c + (0.5 if c>0 else -0.3)
            self.Q[self.last_action] = self.Q.get(self.last_action,0)*0.9 + r
            self.T[(self.last_state,self.last_action)] = state

        self.visits[state] = self.visits.get(state,0)+1

        scored = []

        for a in actions:
            s = self.Q.get(a,0)

            if (state,a) not in self.T:
                s += 1

            ns = self.T.get((state,a))
            if ns:
                s -= 0.1*self.visits.get(ns,0)
                if ns == state:
                    s -= 2
                s += 0.5*self.rollout(ns,actions)

            scored.append((s,a))

        scored.sort(reverse=True)

        eps = max(0.05, 1/(1+self.visits[state]))

        if random.random() < eps:
            action = random.choice(actions)
        else:
            action = scored[0][1]

        self.last_state = state
        self.last_action = action

        if action == "ACTION6":
            h = len(grid)
            w = len(grid[0])
            pts = [(w//2,h//2),(0,0),(w-1,h-1)]
            x,y = random.choice(pts) if random.random()<0.7 else (random.randint(0,w-1),random.randint(0,h-1))
            return {"action":action,"x":x,"y":y}

        return action

# RUN

agent = FinalAgent()

print("Running FINAL DSL agent...")

try:
    run_agent(DummySwarm(agent))
except KeyboardInterrupt:
    print("Finished.")

print("DONE.")

# REQUIRED FILE

df = pd.DataFrame({"dummy": [0]})
df.to_parquet("submission.parquet")

print("'submission.parquet' created.")

Repo path added.
Running FINAL DSL agent...
Running simplified ARC loop...
Finished dummy run.
Finished.
DONE.
'submission.parquet' created.
